# Fitting toric spine cable-graph models

I found that qst=0.5 and mcst=5 works well for all toric spine meshes so far.

## Setup

In [1]:
from mascaf import *
from swctools import SWCModel, FrustaSet, PointSet, plot_model
import logging

logging.basicConfig(level=logging.INFO)
import os

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Load Mesh

In [2]:
spine_idx = 1
mcf_qst = 0.5
mcf_mcst = 5
obj_name = f"TS{spine_idx}"
polylines_name = f"TS{spine_idx}_qst{mcf_qst}_mcst{mcf_mcst}"
mm = MeshManager(mesh_path=f"../data/mesh/processed/{obj_name}.obj")
raw_skeleton = SkeletonGraph.from_txt(
    f"../data/mcf_skeletons/{polylines_name}.polylines.txt"
)
raw_skeleton.prune_short_branches_inplace(min_length_percentile=20)
mm.visualize_mesh_3d(skel=raw_skeleton)

INFO:mascaf.mesh:Loaded mesh: 2378 vertices, 4788 faces


## Skeleton optimization

In [ ]:
opts = SkeletonOptimizerOptions(
    max_iterations=10,
    step_size=2.0,
    smoothing_weight=0.1,
    preserve_terminal_nodes=True,
    preserve_branch_nodes=False,
    n_rays=12,
    verbose=True,
)
print("\nOptimizing skeleton (serial)...")
optimizer = SkeletonOptimizer(raw_skeleton, mm.mesh, opts)
optimized_skeleton = optimizer.optimize()
f = mm.visualize_mesh_3d(
    skel=[raw_skeleton, optimized_skeleton],
    skel_color=["crimson", "blue"],
    show_axes=False,
)
f.show()

skeleton = optimized_skeleton

INFO:mascaf.skeleton_optimizer:Surface crossing detected: 11/363 nodes outside mesh (max distance: 11.0686)



Optimizing skeleton (serial)...


INFO:mascaf.skeleton_optimizer:Starting skeleton optimization...
INFO:mascaf.skeleton_optimizer:  Nodes: 363
INFO:mascaf.skeleton_optimizer:  Max iterations: 20
INFO:mascaf.skeleton_optimizer:  Step size: 2.0000
INFO:mascaf.skeleton_optimizer:  Smoothing weight: 0.1000
INFO:mascaf.skeleton_optimizer:  Iteration 0: avg movement = 1.779615
INFO:mascaf.skeleton_optimizer:  Iteration 10: avg movement = 1.712208
INFO:mascaf.skeleton_optimizer:Surface crossing detected: 1/363 nodes outside mesh (max distance: 1.4166)
INFO:mascaf.skeleton_optimizer:Optimization complete


In [22]:
max_edge_length = 200

swc_out_dir = f"../data/swc/current/{polylines_name}"

# check if directory exists, if not create it
if not os.path.exists(swc_out_dir):
    os.makedirs(swc_out_dir)

radius_strategy = "equivalent_area"
print(f"Computing skeleton for radius_strategy={radius_strategy} ...", end="")
morph = fit_morphology(
    mm.mesh,
    skeleton,
    options=FitOptions(
        max_edge_length=max_edge_length,
        radius_strategy=radius_strategy,
        snap_polylines_to_mesh=True,
    ),
)
# write swc to file
morph.to_swc_file(f"{swc_out_dir}/TS{spine_idx}_s{max_edge_length}_{radius_strategy}.swc")
# validation
validator = Validation(mm, skeleton, morph)
validator.full_validation()


Computing skeleton for radius_strategy=equivalent_area ...

INFO:mascaf.graph_fitting:Tracing done: nodes=24, edges=23, samples=36, section=0, fallback=0 (0.0%)
INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1509 vertices, 3022 faces
INFO:mascaf.validation:  Skeleton: 188 nodes, 187 edges
INFO:mascaf.validation:  MorphologyGraph: 24 nodes, 23 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       11801286.3070
INFO:mascaf.validation:---- Morphology volume: 27583768.7753
INFO:mascaf.validation:---- Ratio:             2.3374
INFO:mascaf.validation:---- Error:             15782482.4683
INFO:mascaf.validation:---- Relative error:    133.74%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         548029.1969
INFO:mascaf.validation:---- Morphology area:   887070.9534
INFO:mascaf.validation:---- Ratio:             1.6187
INFO:mascaf.validation:----

In [23]:
# plot using swctools
make_html = True

swc_filepath = f"{swc_out_dir}/TS{spine_idx}_s{max_edge_length}_{radius_strategy}.swc"
model = SWCModel.from_swc_file(swc_filepath)
model.print_attributes(node_info=False, edge_info=False)
frusta = FrustaSet.from_swc_model(model)
title = f"TS{spine_idx}_s{max_edge_length}_{radius_strategy}"
fig = plot_model(swc_model=model, frusta=frusta, slider=True, title=title)
fig.show()
if make_html:
    fig.write_html(f"../viz/TS{spine_idx}_s{max_edge_length}_{radius_strategy}.html")

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=24 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_parse_result records=24 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_swc_file built nodes=24 edges=23 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=23 sides=16 end_caps=False verts=736 faces=736
INFO:swctools.geometry:FrustaSet.from_swc_model edges=23 sides=16 end_caps=False
INFO:swctools.geometry:batch_frusta count=23 sides=16 end_caps=False verts=736 faces=736
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.0
INFO:swctools.geometry:batch_frusta count=23 sides=16 end_caps=False verts=736 faces=736
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.05
INFO:swctools.geometry:batch_frusta count=23 sides=16 end_caps=False verts=736 faces=736
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.1
INFO:swctools.geometry:batch_frusta count=23

SWCModel: nodes=24, edges=23, components=1, cycles=0, branch_points=5, roots=1, leaves=6, self_loops=0, density=0.0833


INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.85
INFO:swctools.geometry:batch_frusta count=23 sides=16 end_caps=False verts=736 faces=736
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.9
INFO:swctools.geometry:batch_frusta count=23 sides=16 end_caps=False verts=736 faces=736
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.95
INFO:swctools.viz:plot_model slider=True frusta=23 radius_scale_range=[0.0,1.0]


## Tweak radii to fit total surface area

In [24]:
morph = fit_morphology(
    mm.mesh,
    skeleton,
    options=FitOptions(
        max_edge_length=max_edge_length,
        radius_strategy=radius_strategy,
        snap_polylines_to_mesh=False,
    ),
)

morph.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)

# save normalized to file
swc_filepath = (
    f"{swc_out_dir}/TS{spine_idx}_s{max_edge_length}_{radius_strategy}_SAnorm.swc"
)
morph.to_swc_file(swc_filepath)

# load and plot
swc_model = SWCModel.from_swc_file(swc_filepath)
swc_model.print_attributes(node_info=False, edge_info=False)
frusta = FrustaSet.from_swc_model(swc_model)
title = f"TS{spine_idx}_s{max_edge_length}_{radius_strategy}"
fig = plot_model(swc_model=swc_model, frusta=frusta, slider=True, title=title)
fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:mascaf.graph_fitting:Tracing done: nodes=17, edges=16, samples=29, section=0, fallback=0 (0.0%)
INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=17 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_parse_result records=17 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_swc_file built nodes=17 edges=16 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512
INFO:swctools.geometry:FrustaSet.from_swc_model edges=16 sides=16 end_caps=False
INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.0
INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.05
INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512


SWCModel: nodes=17, edges=16, components=1, cycles=0, branch_points=5, roots=1, leaves=6, self_loops=0, density=0.1176


INFO:swctools.viz:plot_model slider=True frusta=16 radius_scale_range=[0.0,1.0]


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1509 vertices, 3022 faces
INFO:mascaf.validation:  Skeleton: 188 nodes, 187 edges
INFO:mascaf.validation:  MorphologyGraph: 17 nodes, 16 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       11801286.3070
INFO:mascaf.validation:---- Morphology volume: 14118140.5107
INFO:mascaf.validation:---- Ratio:             1.1963
INFO:mascaf.validation:---- Error:             2316854.2037
INFO:mascaf.validation:---- Relative error:    19.63%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         548029.1969
INFO:mascaf.validation:---- Morphology area:   548029.1969
INFO:mascaf.validation:---- Ratio:             1.0000
INFO:mascaf.validation:---- Error:             -0.0000
INFO:mascaf.validation:---- Relative error:    -0.00%
INFO:mascaf.validatio